<a href="https://colab.research.google.com/github/eugeniaapvt/Sakila-Data-Analysis/blob/main/Text_M0724025_Eugenia_Bening_Sekar_Pavita.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama  : Eugenia Bening Sekar Pavita

NIM   : M0724025

Kelas : A

Dataset: https://www.kaggle.com/datasets/tirendazacademy/fifa-world-cup-2022-tweets

Dataset ini berisi kumpulan tweet Inggris yang mengandung hashtag #WorldCup2022. Dataset ini diambil dengan scraping data Twitter. Tujuan processing data ini untuk analisis sentimen.

# Loading Dataset

In [ ]:
# import library
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

os.chdir("/content/drive/MyDrive")
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive


In [ ]:
df = pd.read_csv("Dataset MBD/fifa_world_cup_2022_tweets.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22524 entries, 0 to 22523
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       22524 non-null  int64 
 1   Date Created     22524 non-null  object
 2   Number of Likes  22524 non-null  int64 
 3   Source of Tweet  22524 non-null  object
 4   Tweet            22524 non-null  object
 5   Sentiment        22524 non-null  object
dtypes: int64(2), object(4)
memory usage: 1.0+ MB


Dataset ini berisi 22524 baris dan 6 kolom. Hanya kolom `Tweet` yang bertipe object akan digunakan untuk preprocessing data.

In [ ]:
# melihat isi kolom 'Tweet'
df['Tweet'].head(10)

,Tweet
0,What are we drinking today @TucanTribe \n@MadB...
1,Amazing @CanadaSoccerEN #WorldCup2022 launch ...
2,Worth reading while watching #WorldCup2022 htt...
3,Golden Maknae shinning bright\n\nhttps://t.co/...
4,"If the BBC cares so much about human rights, h..."
5,"And like, will the mexican fans be able to scr..."
6,Look like a only me and the Jamaican football ...
7,Really? Football on a Monday morning at 9 and ...
8,"As the World Cup starts in Qatar, it’s Black A..."
9,#WorldCup2022 @ITVSport &amp; @LFSYSTEMMUSIC g...


# Text Preprocessing

**Cleaning Data**

In [ ]:
# lowercase
df['lowercase'] = df['Tweet'].str.lower()

df[['Tweet','lowercase']].sample(10)

,Tweet,lowercase
11474,Flo Rida has written the Qatar national anthem...,flo rida has written the qatar national anthem...
6536,Qatar getting destroyed #WorldCup2022,qatar getting destroyed #worldcup2022
5634,Enner Valencia becomes the first goal-scorer o...,enner valencia becomes the first goal-scorer o...
19999,Who will win the World Cup?…👇🏻\n\n🤔 #WorldCup2022,who will win the world cup?…👇🏻\n\n🤔 #worldcup2022
6371,Back in business Equador #WorldCup2022,back in business equador #worldcup2022
11877,Anyone purchasing a #WorldCup2022 ticket shoul...,anyone purchasing a #worldcup2022 ticket shoul...
13199,@AlexGangueRuzic Yeahhh. I watched the intervi...,@alexgangueruzic yeahhh. i watched the intervi...
18507,For every messi goal scored in this tournament...,for every messi goal scored in this tournament...
7641,Finally Start's 🥳#FootballWorldCup2022 @FIFAWo...,finally start's 🥳#footballworldcup2022 @fifawo...
18187,@AlJazeera You're right now indulging in human...,@aljazeera you're right now indulging in human...


Format teks pada kolom `Tweet` masih tidak konsisten. Perbedaan huruf besar dan huruf kecil pada kalimat menyulitkan tahapan preprocessing selanjutnya. Maka dari itu, perlu penyeragaman format menjadi lowercase. Hasil data yang sudah diubah dapat dilihat di kolom `lowercase`.

In [ ]:
# remove URL
df['remove_url'] = df['lowercase'].str.replace(r'http\S+|www\S+|https\S+', '', regex=True)

df[['lowercase','remove_url']].sample(10)

,lowercase,remove_url
8034,no offside to be found... #worldcup2022 https:...,no offside to be found... #worldcup2022
9328,"the day that was too far away, it's today ..\n...","the day that was too far away, it's today ..\n..."
21032,some quick bytes before tonight's #fifaworldcu...,some quick bytes before tonight's #fifaworldcu...
11000,the world cup starts now!!! #worldcup2022,the world cup starts now!!! #worldcup2022
16114,“world cup + argentina” #mexcworldcup01 #world...,“world cup + argentina” #mexcworldcup01 #world...
14504,looking forward to watch qatar v ecuador #worl...,looking forward to watch qatar v ecuador #worl...
10967,our short film 👇\nhttps://t.co/7tsou3npqc\n\n#...,our short film 👇\n\n\n#pspk\n#fifaworldcup #be...
3603,how do we think this second half is going to g...,how do we think this second half is going to g...
16248,obviously nobody told the quataris what #malum...,obviously nobody told the quataris what #malum...
14192,the #worldcup2022 about to kick off.\n\nwho yo...,the #worldcup2022 about to kick off.\n\nwho yo...


Menghapus URL saat preprocessing teks penting karena link tidak membawa makna linguistik yang relevan dan hanya menambah noise pada data. URL juga sangat variatif sehingga memperbesar vocabulary secara tidak perlu, mengganggu tokenisasi, dan dapat menyebabkan sparsity pada representasi teks seperti TF-IDF atau Bag-of-Words. Hasil data yang sudah diremove URL dapat dilihat di kolom `remove_url`.

In [ ]:
# menghapus @username, hashtag, dan emoji
def clean_text(text):
  # remove @username & hashtag
  text = re.sub(r"@\w+|#\w+", "", text)
  # remove emoji
  text = text.encode("ascii", "ignore").decode("ascii")
  # remove numbers
  text = re.sub(r"\d+", "", text)
  # remove punctuation
  text = text.translate(str.maketrans("", "", string.punctuation))
  # remove extra spaces
  text = re.sub(r"\s+", " ", text).strip()
  return text

df["clean_text"] = df["remove_url"].apply(clean_text)

df[['remove_url','clean_text']].sample(10)

,remove_url,clean_text
5106,that's how you counter a fix #worldcup2022,thats how you counter a fix
4008,the ecuador crowd is hyped 🇪🇨 #fifaworldcup\n...,the ecuador crowd is hyped
4036,\n\nsneak peak on how the @footballqatar team ...,sneak peak on how the team was formed
5278,wow is this boring 😴 #worldcup2022,wow is this boring
15410,i am watching what i thought was world cup pre...,i am watching what i thought was world cup pre...
19810,@zinnianetwork @crew3xyz $zinn #zinnianetwork ...,zinn
449,@garylineker your disgusting #doublestandards ...,your disgusting and unbelievable are literally...
561,"amazing artists, you did amazing, you guys did...",amazing artists you did amazing you guys did m...
11551,let’s go⚽️🏆💪🏼#worldcup2022,lets go
12280,man this @espnfc juz hate #portugal win world ...,man this juz hate win world cup never ever see...


Fungsi `clean_text` berguna untuk membersihkan data. Pertama, fungsi menghapus @username dan hashtag karena elemen tersebut merupakan metadata media sosial yang tidak menambah makna linguistik sehingga hanya menjadi noise dalam data. Selanjutnya, emoji dihilangkan karena tidak termasuk karakter teks standar dan dapat mengganggu proses tokenisasi. Fungsi kemudian menghapus angka karena nilai numerik sering tidak relevan dalam pemrosesan bahasa, terutama ketika konteks kalimat lebih penting dibandingkan angka itu sendiri. Setelah itu, tanda baca seperti titik, koma, tanda tanya, dan simbol lainnya dihapus agar teks menjadi lebih konsisten dan mudah diproses oleh model NLP. Terakhir, fungsi menghapus spasi berlebih agar hasil akhir lebih rapi dan tidak menimbulkan token kosong. Data yang sudah bersih dapat dilihat pada kolom `clean_text`

**Mapping Slang Words**

In [ ]:
slang_words= {                  # slang mapping ini dibuat berdasarkan slang yang biasanya digunakan dalam tweet sepak bola
    "lol": "laugh out loud",
    "ppl": "people",
    "r": "are",
    "u": "you",
    "omg": "oh my god",
    "lmao": "laughing my ass off",
    "lmaooooo": "laughing my ass off",
    "gm": "good morning",
    "2moro": "tomorrow",
    "2nite": "tonight",
    "soooo": "so",
    "gg": "good game",
    "goat": "greatest of all time",
    "goated": "performed outstandingly",
    "clutch": "decisive under pressure",
    "bottle": "fail under pressure",
    "bottled": "failed under pressure",
    "bottling": "failing under pressure",
    "frauds": "overrated players",
    "fraud": "overrated player",
    "cold": "very skillful",
    "cook": "dominate",
    "cooked": "dominated",
    "cooking": "dominating",
    "saucy": "stylish or skillful",
    "brace": "two goals",
    "sitter": "easy scoring chance",
    "clearance": "defensive removal",
    "xg": "expected goals",
    "pen": "penalty",
    "itscominghome": "england will win the world cup",
    "vamos": "let's go",
    "vamos argentina": "go argentina",
    "vamos messi": "go messi",
    "alhamdulillah": "thank god",
    "subhanallah": "glory to god",
    "inshallah": "god willing",
    "lfg": "let's freaking go",
    "vamosss": "let's go",
    "vamosssss": "let's go",
    "vamosss argentina": "go argentina",
    "vamosss messi": "go messi",
    "hattrick": "three goals",
    "banger": "amazing goal",
    "top bins": "perfect shot into the top corner",
    "robbed": "unfairly treated",
    "rigged": "match is manipulated",
    "fixed": "match is manipulated",
    "varball": "overuse of video assistant referee",
    "bald fraud": "bad coach",
    "scripted": "predetermined outcome",
    "disasterclass": "very bad performance",
    "masterclass": "outstanding performance",
    "insane": "amazing",
    "crazy": "amazing",
    "peak": "excellent moment",
    "mid": "boring or average",
    "trash": "terrible",
    "washed": "past their prime",
    "w": "win or good",
    "l": "loss or bad",
    "dub": "win",
    "dubbed": "won",
    "w squad": "good team",
    "l squad": "bad team",
    "qatarball": "qatar football style",
    "desert penalty": "questionable penalty during qatar matches",
    "ghost goal": "goal not counted",
    "dyk": "did you know",
    "fab": "fabulous",
    "fking": "fucking"
}

def map_slang_words(text):
    words = text.split()
    mapped_words = [slang_words.get(word, word) for word in words]
    return " ".join(mapped_words)

df['mapped_slang_words'] = df['clean_text'].apply(map_slang_words)

df[['clean_text', 'mapped_slang_words']].sample(10)

,clean_text,mapped_slang_words
11061,this if youre eligible do the double book your...,this if youre eligible do the double book your...
20507,breaking france striker karim benzema will mis...,breaking france striker karim benzema will mis...
3107,are you watching the opening game keep an eye ...,are you watching the opening game keep an eye ...
6360,first goal of this world cup is penalty by qat...,first goal of this world cup is penalty by qat...
8041,no goal what the hell,no goal what the hell
12741,any thoughts,any thoughts
7747,corruption in plain daylight,corruption in plain daylight
6347,and this brought him back jimin came home,and this brought him back jimin came home
7869,world cup rigged ecuador robbed,world cup match is manipulated ecuador unfairl...
13049,where do the gay dancers stay or are they stra...,where do the gay dancers stay or are they stra...


Fungsi mapping slang word ini digunakan untuk mengubah kata-kata slang menjadi kata yang lebih baku. Tujuannya agar teks menjadi lebih konsisten dan mudah diproses pada analisis selanjutnya. Kolom baru `mapped_slang_words` menampilkan hasil teks yang sudah dinormalisasi sehingga tidak lagi mengandung slang yang dapat mengganggu akurasi model.

**Remove Stopwords**

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered = [w for w in words if w.lower() not in stop_words]
    return " ".join(filtered)

df['remove_stopwords_text'] = df['mapped_slang_words'].apply(remove_stopwords)

df[['clean_text', 'remove_stopwords_text']].sample(10)

,clean_text,remove_stopwords_text
11888,beautiful way to start my sunday,beautiful way start sunday
20110,happy world cup day make sure youre in the kno...,happy world cup day make sure youre know read ...
15467,fifa world cup opening ceremony live streams o...,fifa world cup opening ceremony live streams o...
13034,is the best show was stunning and meaningful,best show stunning meaningful
16408,i hate that tukoh taka song sm like what bring...,hate tukoh taka song sm like bring shakira spe...
3709,mexico is in the house ready for the party qatar,mexico house ready party qatar
17672,you dont have to be a football expert to parti...,dont football expert participate poloniex worl...
10674,no comments for now re appropriateness of qata...,comments appropriateness qatar host going happ...
12608,real supporters,real supporters
2554,thats just a tease budweiser ads at a no beer ...,thats tease budweiser ads beer world cup game


Fungsi `remove_stopwords` dipakai untuk menghapus kata-kata umum (stopwords) seperti the, is, at, a, an, of yang tidak membawa makna penting dalam analisis sentimen. Kode mengambil daftar stopwords bahasa Inggris dari NLTK lalu memfilter setiap kata dalam teks. Hasil akhirnya disimpan dalam kolom baru `remove_stopwords_text`.

**Lemmatization**

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

wnl = WordNetLemmatizer()

def lemmatize_text(text):
    return " ".join(wnl.lemmatize(word) for word in text.split())

df['lemmatize_text'] = df['remove_stopwords_text'].apply(lemmatize_text)

df[['remove_stopwords_text', 'lemmatize_text']].sample(10)

,remove_stopwords_text,lemmatize_text
15668,country banning alcohol ariel view stadium qui...,country banning alcohol ariel view stadium qui...
18727,thanks request passed traders well touch price...,thanks request passed trader well touch price ...
14120,good luck boys,good luck boy
3942,mean coach qatar eric ten hag expect,mean coach qatar eric ten hag expect
6126,clearly referee cheque didnt clear time must b...,clearly referee cheque didnt clear time must b...
1512,take knee world cup opener iran although domes...,take knee world cup opener iran although domes...
19796,feels like shisha like fifa,feel like shisha like fifa
17843,world cup england tickets available please dm,world cup england ticket available please dm
10129,minutes world cup theres already goal madness ...,minute world cup there already goal madness ec...
4246,thats goals given lets name former vice presid...,thats goal given let name former vice presiden...


Fungsi lemmatize_text digunakan untuk mengubah tiap kata ke bentuk dasarnya (lemma) menggunakan WordNetLemmatizer. Hasil akhirnya disimpan dalam kolom `lemmatize_text`.

Sebelumnya saya sudah mencoba menggunakan stemming, namun karena hasil stemming terlalu kasar jadi lemmatize saya pilih karena menghasilkan bentuk linguistik yang lebih rapi.

Dalam tujuan saya yaitu analisis sentimen pada tweet World Cup 2022, lemmatization lebih cocok karena menjaga makna kata agar nuansa emosinya tidak banyak berubah/terpotong.

In [ ]:
def remove_short_words(text):
    return ' '.join([word for word in text.split() if len(word) > 2])

df['lemmatize_text_cleaned'] = df['lemmatize_text'].apply(remove_short_words)

df[['lemmatize_text', 'lemmatize_text_cleaned']].sample(10)

,lemmatize_text,lemmatize_text_cleaned
4786,superb cross brilliant header good player long...,superb cross brilliant header good player long...
18565,sure team perform year havent really followed ...,sure team perform year havent really followed ...
12479,minute thing get started terrible skillful fee...,minute thing get started terrible skillful fee...
16824,hi available player requestabet world cup mark...,available player requestabet world cup market ...
5709,match feel like elder sister bullying younger ...,match feel like elder sister bullying younger ...
22410,official tomorrow fifa world cup qatar change ...,official tomorrow fifa world cup qatar change ...
21952,death nepali worker attributed heat exposure r...,death nepali worker attributed heat exposure r...
13757,made,made
3570,nearly,nearly
21407,even start tie rank netherland denmark great p...,even start tie rank netherland denmark great p...


Sebagian besar kata-kata yang panjangnya 1 atau 2 karakter biasanya tidak terlalu signifikan untuk analisis sentimen, sehingga dihapus untuk mengurangi noise.

**Tokenization**

In [ ]:
from nltk.tokenize import word_tokenize

def tokenize_text(text):
    return word_tokenize(text)

df['tokenized_text'] = df['lemmatize_text_cleaned'].apply(tokenize_text)

df[['lemmatize_text_cleaned', 'tokenized_text']].sample(10)


,lemmatize_text_cleaned,tokenized_text
22102,online betting william hill bet get promo code...,"[online, betting, william, hill, bet, get, pro..."
3729,goal scored scored west ham player massive,"[goal, scored, scored, west, ham, player, mass..."
10734,let begin,"[let, begin]"
17067,ran mile nike run club getting one bright skil...,"[ran, mile, nike, run, club, getting, one, bri..."
1923,got parlay thats one good thing,"[got, parlay, thats, one, good, thing]"
4953,continent win,"[continent, win]"
9254,bro theyre already rigging,"[bro, theyre, already, rigging]"
1458,south africa world cup still best world cup ha...,"[south, africa, world, cup, still, best, world..."
260,lot right wing flag account pop kneeling footb...,"[lot, right, wing, flag, account, pop, kneelin..."
1856,come world cup brilliant speaking position rig...,"[come, world, cup, brilliant, speaking, positi..."


Tokenization dilakukan untuk memecah kalimat menjadi unit kata/token sehingga memudahkan proses model NLP. Perbandingan sebelum dan sesudah tokenization dapat dilihat pada kolom `lemmatize_text_cleaned` dan `tokenized_text`

# Text Vectorization

**Bag of Words**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

documents = df['tokenized_text'].astype(str).tolist()

X = vectorizer.fit_transform(documents)

print(X.toarray())
print(vectorizer.get_feature_names_out())

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
['abbas' 'abeg' 'ability' ... 'zakumi' 'zero' 'zone']


Model Bag of Words (BoW) pada output tersebut menunjukkan bahwa metode ini mampu mengekstraksi kata-kata penting dari kumpulan tweet dan mengubahnya menjadi fitur numerik yang dapat digunakan untuk analisis lebih lanjut. Kelebihannya, BoW sangat sederhana, mudah diterapkan, dan efektif untuk merepresentasikan frekuensi kemunculan kata seperti “ability”, “zero”, “zone”, dan lainnya, terutama setelah proses pembersihan data yang baik. Namun, output matriks yang didominasi angka nol mencerminkan salah satu kelemahan utama BoW, yaitu sifatnya yang sparse banyak dokumen tidak mengandung sebagian besar kata dalam kosakata. Selain itu, BoW tidak mempertimbangkan konteks atau urutan kata, sehingga tidak dapat menangkap makna semantik atau hubungan antarkata. BoW efektif untuk representasi sederhana tetapi terbatas dalam memahami makna mendalam dari teks.

**TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = df['tokenized_text'].astype(str).tolist()

X_tfidf = tfidf_vectorizer.fit_transform(documents)

print(X_tfidf.toarray())
print(tfidf_vectorizer.get_feature_names_out())

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
['abbas' 'abeg' 'ability' ... 'zakumi' 'zero' 'zone']


TF-IDF adalah perluasan BoW yang memberi bobot pada frekuensi kata berdasarkan kepentingannya di seluruh dokumen. TF-IDF memiliki beberapa kelebihan, di antaranya sederhana dan efisien karena mudah dihitung dengan sumber daya komputasi yang minimal, mampu memberikan bobot penting pada kata dengan menurunkan pengaruh kata-kata yang terlalu umum, serta efektif untuk information retrieval karena membantu meningkatkan peringkat relevansi dokumen dalam pencarian. Namun, TF-IDF juga memiliki kelemahan seperti menghasilkan vektor berdimensi tinggi dan bersifat sparse sehingga membutuhkan memori dan komputasi lebih besar, tidak mampu menangkap konteks atau makna kata karena hanya berbasis frekuensi, serta memperlakukan sinonim sebagai kata yang berbeda sehingga dapat menurunkan akurasi model.